In [1]:
import muon as mu
import numpy as np
import pandas as pd
import anndata
from scipy.sparse import csr_matrix

# Noise (subset proteins, flip values) and redundancy

'''
(1) noise — per cell this randomly subset which proteins are "measured" (variable panel size, ~1–257 proteins) and flips ~10% of the remaining values. 
(2) redundancy — this generates K independent noisy paragraphs per cell, so each cell has multiple views that can be
  used as positive pairs. Output is a CSV (cell_texts_augmented_K{K}_flip{FLIP_PROB}.csv) with columns cell_id, version, subset_size, text.
'''

DATA_PATH = '/home/lemanic-g04/'
SEED = 42
K = 5                          # number of paragraphs per cell
FLIP_PROB = 0.10
MIN_PROTEINS = 1
MAX_PROTEINS = 257
SHUFFLE_ORDER = True           # also randomize protein order in the sentence

TEMPLATES = [
  ("This cell expresses the following surface proteins: {pos}. However, it does not express {neg}.",
   "This cell does not express any of the measured surface proteins. Specifically, it lacks expression of {neg}.",
   "This cell expresses all of the measured surface proteins. Specifically, it expresses {pos}."),

  ("The proteins expressed in this cell are: {pos}. It does not express {neg}.",
   "None of the measured proteins are expressed in this cell: {neg}.",
   "All measured proteins are expressed in this cell: {pos}."),

  ("Among the measured surface proteins, this cell is positive for {pos}. It is negative for {neg}.",
   "This cell is negative for all measured surface proteins: {neg}.",
   "This cell is positive for all measured surface proteins: {pos}."),

  ("Surface protein analysis shows expression of {pos}. No expression was detected for {neg}.",
   "No surface protein expression was detected. Absent proteins include {neg}.",
   "Expression was detected for all measured surface proteins: {pos}."),
]

rng = np.random.default_rng(SEED)

# Load clean binarized data
mdata = mu.read(DATA_PATH + 'GSE194315_binarized_with_text_mu.h5mu')
protein = mdata.mod['protein']
X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
X = X.astype(bool)
protein_names = protein.var_names.to_numpy()
n_cells, n_proteins = X.shape

def humanize(names):
  names = list(names)
  if len(names) == 0: return ""
  if len(names) == 1: return names[0]
  if len(names) == 2: return f"{names[0]} and {names[1]}"
  return ", ".join(names[:-1]) + f", and {names[-1]}"

'''
def cell_to_text(row, mask):
  expressed     = protein_names[row & mask]
  not_expressed = protein_names[(~row) & mask]
  if SHUFFLE_ORDER:
      expressed     = rng.permutation(expressed)
      not_expressed = rng.permutation(not_expressed)
  if len(expressed) == 0 and len(not_expressed) == 0:
      return "No surface proteins were measured for this cell."
  if len(expressed) == 0:
      return (f"This cell does not express any of the measured surface proteins. "
              f"Specifically, it lacks expression of {humanize(not_expressed)}.")
  if len(not_expressed) == 0:
      return (f"This cell expresses all of the measured surface proteins. "
              f"Specifically, it expresses {humanize(expressed)}.")
  return (f"This cell expresses the following surface proteins: {humanize(expressed)}. "
          f"However, it does not express {humanize(not_expressed)}.")
'''
def cell_to_text(row, mask):
  expressed     = protein_names[row & mask]
  not_expressed = protein_names[(~row) & mask]
  if SHUFFLE_ORDER:
      expressed     = rng.permutation(expressed)
      not_expressed = rng.permutation(not_expressed)

  both_tmpl, neg_only_tmpl, pos_only_tmpl = TEMPLATES[rng.integers(len(TEMPLATES))]

  if len(expressed) == 0 and len(not_expressed) == 0:
      return "No surface proteins were measured for this cell."
  if len(expressed) == 0:
      return neg_only_tmpl.format(neg=humanize(not_expressed))
  if len(not_expressed) == 0:
      return pos_only_tmpl.format(pos=humanize(expressed))
  return both_tmpl.format(pos=humanize(expressed), neg=humanize(not_expressed))
    
# Generate K paragraphs per cell
records = []
for k in range(K):
  print(f"Generating version {k+1}/{K}")
  for i in range(n_cells):
      # Per-cell subset
      size = int(rng.integers(MIN_PROTEINS, MAX_PROTEINS + 1))
      keep_idx = rng.choice(n_proteins, size=size, replace=False)
      mask = np.zeros(n_proteins, dtype=bool)
      mask[keep_idx] = True

      # Per-cell flips
      flips = (rng.random(n_proteins) < FLIP_PROB) & mask
      row_noisy = X[i] ^ flips

      # Text
      text = cell_to_text(row_noisy, mask)

      records.append({
          'cell_id': protein.obs_names[i],
          'version': k,
          'subset_size': size,
          'text': text,
      })

df = pd.DataFrame(records)
print(f"Generated {len(df)} paragraphs ({n_cells} cells × {K} versions)")

# Save
out_csv = f'cell_texts_augmented_K{K}_flip{int(FLIP_PROB*100)}.csv'
df.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")

/home/gnoto_venvs/text/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gnoto_venvs/text/lib/python3.12/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1272: Fut

Generating version 1/5
Generating version 2/5
Generating version 3/5
Generating version 4/5
Generating version 5/5
Generated 903970 paragraphs (180794 cells × 5 versions)
Saved cell_texts_augmented_K5_flip10.csv


In [2]:
# Noise (subset proteins) and redundancy (different paragraphs per cell based on orderings)

'''
  (1) subsetting — per cell this randomly subsets which proteins are "measured" (variable panel size, ~1–257 proteins).
      Values are NOT flipped — the biology is preserved, only the observable panel changes.
  (2) redundancy — this generates K independent subsetted paragraphs per cell, so each cell has multiple views that can be
      used as positive pairs. Output is a CSV (cell_texts_augmented_K{K}_subsetOnly.csv) with columns cell_id, version, subset_size, text.
  '''

DATA_PATH = '/home/lemanic-g04/'
SEED = 42
K = 5                          # number of paragraphs per cell
MIN_PROTEINS = 1
MAX_PROTEINS = 257
SHUFFLE_ORDER = True           # also randomize protein order in the sentence

TEMPLATES = [
  ("This cell expresses the following surface proteins: {pos}. However, it does not express {neg}.",
   "This cell does not express any of the measured surface proteins. Specifically, it lacks expression of {neg}.",
   "This cell expresses all of the measured surface proteins. Specifically, it expresses {pos}."),

  ("The proteins expressed in this cell are: {pos}. It does not express {neg}.",
   "None of the measured proteins are expressed in this cell: {neg}.",
   "All measured proteins are expressed in this cell: {pos}."),

  ("Among the measured surface proteins, this cell is positive for {pos}. It is negative for {neg}.",
   "This cell is negative for all measured surface proteins: {neg}.",
   "This cell is positive for all measured surface proteins: {pos}."),

  ("Surface protein analysis shows expression of {pos}. No expression was detected for {neg}.",
   "No surface protein expression was detected. Absent proteins include {neg}.",
   "Expression was detected for all measured surface proteins: {pos}."),
]

rng = np.random.default_rng(SEED)

# Load clean binarized data
mdata = mu.read(DATA_PATH + 'GSE194315_binarized_with_text_mu.h5mu')
protein = mdata.mod['protein']
X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
X = X.astype(bool)
protein_names = protein.var_names.to_numpy()
n_cells, n_proteins = X.shape

def humanize(names):
  names = list(names)
  if len(names) == 0: return ""
  if len(names) == 1: return names[0]
  if len(names) == 2: return f"{names[0]} and {names[1]}"
  return ", ".join(names[:-1]) + f", and {names[-1]}"

def cell_to_text(row, mask):
  expressed     = protein_names[row & mask]
  not_expressed = protein_names[(~row) & mask]
  if SHUFFLE_ORDER:
      expressed     = rng.permutation(expressed)
      not_expressed = rng.permutation(not_expressed)

  both_tmpl, neg_only_tmpl, pos_only_tmpl = TEMPLATES[rng.integers(len(TEMPLATES))]

  if len(expressed) == 0 and len(not_expressed) == 0:
      return "No surface proteins were measured for this cell."
  if len(expressed) == 0:
      return neg_only_tmpl.format(neg=humanize(not_expressed))
  if len(not_expressed) == 0:
      return pos_only_tmpl.format(pos=humanize(expressed))
  return both_tmpl.format(pos=humanize(expressed), neg=humanize(not_expressed))

# Generate K paragraphs per cell
records = []
for k in range(K):
  print(f"Generating version {k+1}/{K}")
  for i in range(n_cells):
      # Per-cell subset
      size = int(rng.integers(MIN_PROTEINS, MAX_PROTEINS + 1))
      keep_idx = rng.choice(n_proteins, size=size, replace=False)
      mask = np.zeros(n_proteins, dtype=bool)
      mask[keep_idx] = True

      # No flips — use the clean values at measured positions
      row_clean = X[i]

      # Text
      text = cell_to_text(row_clean, mask)

      records.append({
          'cell_id': protein.obs_names[i],
          'version': k,
          'subset_size': size,
          'text': text,
      })

df = pd.DataFrame(records)
print(f"Generated {len(df)} paragraphs ({n_cells} cells × {K} versions)")

# Save
out_csv = f'cell_texts_augmented_K{K}_subsetOnly.csv'
df.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")

/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Generating version 1/5
Generating version 2/5
Generating version 3/5
Generating version 4/5
Generating version 5/5
Generated 903970 paragraphs (180794 cells × 5 versions)
Saved cell_texts_augmented_K5_subsetOnly.csv


In [3]:
# 1. Pick a specific cell and show all K versions
test_cell = 'PBMC-02-1_AAACCCAGTAGCTGTT'
cell_rows = df[df['cell_id'] == test_cell]
print(f"=== All {K} versions of {test_cell} ===")
print(cell_rows[['version', 'subset_size']].to_string(index=False))
print(f"All subset_sizes different? {cell_rows['subset_size'].nunique() == K}")
print()

# 2. Show that texts are different across versions for that cell
for _, row in cell_rows.iterrows():
  print(f"--- Version {row['version']} (subset_size={row['subset_size']}) ---")
  print(row['text'][:150] + "...")
  print()

# 3. Compare different cells within the same version (k=0)
k0 = df[df['version'] == 0].head(10)
print(f"=== First 10 cells in version 0 ===")
print(k0[['cell_id', 'subset_size']].to_string(index=False))
print(f"All subset_sizes different? {k0['subset_size'].nunique() == len(k0)}")
print()

# 4. Distribution of subset_size
print(f"=== Global subset_size stats ===")
print(f"Min: {df['subset_size'].min()}, Max: {df['subset_size'].max()}")
print(f"Mean: {df['subset_size'].mean():.1f}, Std: {df['subset_size'].std():.1f}")
print(f"Unique values: {df['subset_size'].nunique()} (expect ~257 if range is 1-257)")
print()

# 5. Check that texts differ between versions of the same cell
print(f"=== Text uniqueness check (sample of 100 cells) ===")
sample_cells = df['cell_id'].unique()[:100]
all_unique = 0
for cid in sample_cells:
  texts = df[df['cell_id'] == cid]['text'].values
  if len(set(texts)) == K:
      all_unique += 1
print(f"{all_unique}/100 cells have {K} fully unique texts across versions")

=== All 5 versions of PBMC-02-1_AAACCCAGTAGCTGTT ===
 version  subset_size
       0           23
       1          198
       2          160
       3          216
       4          232
All subset_sizes different? True

--- Version 0 (subset_size=23) ---
The proteins expressed in this cell are: CD2-1 and GPR56. It does not express TIM-4, CD63-1, CD62P-P-Selectin, CD141-Thrombomodulin, CD83-1, CD54, CD4...

--- Version 1 (subset_size=198) ---
The proteins expressed in this cell are: TCRVdelta2, GPR56, KLRG1-MAFA, CD45RA, and CD3. It does not express integrinbeta7, Notch3, CD307d-FcRL4, TCRV...

--- Version 2 (subset_size=160) ---
This cell expresses the following surface proteins: CD3, TCRVdelta2, CD2-1, KLRG1-MAFA, and CD352-NTB-A. However, it does not express CD133, CD4-1, CD...

--- Version 3 (subset_size=216) ---
This cell expresses the following surface proteins: CD45RA, TCRVdelta2, CD3, GPR56, KLRG1-MAFA, and CD352-NTB-A. However, it does not express CD112-Ne...

--- Version 4 (sub